# 04 · Leakage, deduplication & class crosswalk
### *From Diagnosis to Decision* — ICA 2026

Three checks that protect the validity of every downstream number:

1. **Leaf-grouped split** for PlantVillage using `leaf_id` — the single most
   important guard against inflated accuracy.
2. **Cross-dataset near-duplicate detection** via perceptual hashing — any image
   shared between a *training* set and an *evaluation* set silently invalidates
   the generalization claim.
3. **Class crosswalk** — a seed mapping from each dataset's raw labels to a
   canonical `(crop, disease)` space, so datasets can be compared at all.

> The perceptual-hash section runs locally on whatever is present (PlantDoc by
> default) and demonstrates *intra*-dataset duplicates; the *cross*-dataset
> matrix fills in on Colab once ≥2 datasets are downloaded.

## 1 · Setup

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
# --- Which datasets are actually downloaded? Notebooks adapt to what's present.
import pandas as pd

CANDIDATES = {
    "plantvillage": DATA_RAW / "plantvillage",
    "plantwild":    DATA_RAW / "plantwild",
    "plantseg":     DATA_RAW / "plantseg",
    "plantdoc":     DATA_RAW / "plantdoc",
    "fieldplant":   DATA_RAW / "fieldplant",
    "cassava":      DATA_RAW / "cassava",
    "master":       DATA_RAW / "master_plant_disease",
    "bracol":       DATA_RAW / "bracol",
    "rocole":       DATA_RAW / "rocole",
}
PRESENT = {k: v for k, v in CANDIDATES.items() if v.exists() and any(v.rglob("*"))}
print("Present datasets:", list(PRESENT) or "(none yet — run 00_download_verify first)")

In [ ]:
import numpy as np, pandas as pd, pathlib
from PIL import Image
import imagehash
from collections import defaultdict
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}

def iter_images(root):
    for p in pathlib.Path(root).rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            yield p

## 2 · Leaf-grouped split for PlantVillage (leak-safe)

PlantVillage photographs each physical leaf several times. A random split leaks
one leaf across train/test and can inflate accuracy by double digits. We group
by `leaf_id` so **all** photos of a leaf land in the same fold, then write the
split table that notebook `05` consumes.

In [ ]:
PV = DATA_RAW / "plantvillage" / "hf_arrow"
if PV.exists():
    from datasets import load_from_disk
    from sklearn.model_selection import GroupShuffleSplit
    ds = load_from_disk(str(PV))
    d = ds[list(ds)[0]]
    feats = d.features
    assert "leaf_id" in feats, "This PlantVillage copy lacks leaf_id — re-download from HF (see 00)."
    df = pd.DataFrame({"idx": range(len(d)), "label": d["label"], "leaf_id": d["leaf_id"]})

    # 70/15/15 leaf-grouped
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=0)
    tr_idx, tmp_idx = next(gss.split(df, groups=df["leaf_id"]))
    tmp = df.iloc[tmp_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=0)
    va_rel, te_rel = next(gss2.split(tmp, groups=tmp["leaf_id"]))
    df["split"] = "train"
    df.loc[tmp.iloc[va_rel].index, "split"] = "val"
    df.loc[tmp.iloc[te_rel].index, "split"] = "test"

    # leakage assertion: no leaf_id spans two splits
    spans = df.groupby("leaf_id")["split"].nunique()
    assert (spans == 1).all(), "LEAK: a leaf_id appears in multiple splits!"
    df.to_csv(DATA_INTERIM / "plantvillage_leafgrouped_split.csv", index=False)
    print("leaf-grouped split written. splits:", df["split"].value_counts().to_dict())
    print("unique leaves:", df["leaf_id"].nunique(), "| images:", len(df))

    # For contrast: how optimistic is a RANDOM split? (report this in the paper)
    print("\nWhy it matters: a random split would place ~"
          f"{(df.groupby('leaf_id').size()>1).mean():.0%} of leaves' photos on both sides.")
else:
    print("PlantVillage not present — run 00 on Colab. (leaf_id-based split needs the HF copy.)")

## 3 · Perceptual-hash near-duplicate detection

`phash` (perceptual hash) gives each image a 64-bit fingerprint; Hamming
distance ≤ `THRESH` flags near-duplicates (resizes, re-compressions, crops).
We hash every present dataset, store the index, and report:
- **intra-dataset** duplicates (inflate a single set), and
- **cross-dataset** collisions between train-source and eval sets — the ones
  that break the generalization experiment.

In [ ]:
THRESH = 6   # Hamming distance for "near-duplicate"; 0 = identical

def hash_dataset(name, root, cap_per_dataset=4000):
    rows=[]
    paths = list(iter_images(root))
    if len(paths) > cap_per_dataset:
        paths = list(np.random.default_rng(0).choice(paths, cap_per_dataset, replace=False))
    for p in paths:
        try:
            h = imagehash.phash(Image.open(p).convert("RGB"))
            rows.append((name, str(p), str(h)))
        except Exception:
            pass
    return rows

records=[]
for name, root in PRESENT.items():
    r = hash_dataset(name, root)
    records += r
    print(f"hashed {name}: {len(r)} images")
idx = pd.DataFrame(records, columns=["dataset","path","phash"])
# Parquet on Colab (has pyarrow); CSV fallback so it also runs on a bare local env.
try:
    idx.to_parquet(DATA_INTERIM / "phash_index.parquet")
    print("wrote phash_index.parquet |", end=" ")
except Exception:
    idx.to_csv(DATA_INTERIM / "phash_index.csv", index=False)
    print("pyarrow missing -> wrote phash_index.csv |", end=" ")
print("total hashed:", len(idx))

In [ ]:
# Group exact-phash collisions first (fast), then report cross-dataset ones.
def hamming(a,b): return bin(int(a,16)^int(b,16)).count("1")

# exact duplicates (distance 0)
dup_groups = idx.groupby("phash")["dataset"].agg(list)
exact_multi = dup_groups[dup_groups.map(len) > 1]
intra = sum(1 for v in exact_multi if len(set(v))==1)
cross = sum(1 for v in exact_multi if len(set(v))>1)
print(f"exact-duplicate phash groups: {len(exact_multi)}  (intra-dataset {intra}, cross-dataset {cross})")

# near-duplicate CROSS-dataset scan (only if >=2 datasets present)
if idx["dataset"].nunique() >= 2:
    hits=[]
    by_ds = {k:v.reset_index(drop=True) for k,v in idx.groupby("dataset")}
    names=list(by_ds)
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            A,B = by_ds[names[i]], by_ds[names[j]]
            # cheap: compare only where high hex overlap; full O(n*m) ok at caps
            for _,ra in A.iterrows():
                for _,rb in B.iterrows():
                    if hamming(ra["phash"], rb["phash"]) <= THRESH:
                        hits.append((names[i],ra["path"],names[j],rb["path"]))
    cross_near = pd.DataFrame(hits, columns=["ds_a","path_a","ds_b","path_b"])
    cross_near.to_csv(DATA_INTERIM/"cross_dataset_near_duplicates.csv", index=False)
    print(f"cross-dataset near-duplicates (<= {THRESH}): {len(cross_near)}")
    if len(cross_near):
        print("!! Remove these from the EVAL set before reporting cross-dataset accuracy.")
        display(cross_near.head())
else:
    print("Only one dataset present — cross-dataset scan runs on Colab once >=2 exist.")
    print("Intra-dataset near-duplicates are still worth removing; extend the scan if needed.")

## 4 · Class crosswalk (seed)

To compare datasets you need one label space. This builds a **seed** crosswalk:
`source_dataset, original_label → crop_canonical, disease_canonical`, parsed
heuristically from label strings. **Every unmatched or ambiguous row is flagged
for manual review** rather than guessed — reviewers will not accept auto-mapped
disease labels taken on faith.

In [ ]:
import re
def parse_label(lab):
    s = lab.replace("___","_").replace("__","_").replace("-"," ").strip()
    s = re.sub(r"\s+"," ", s)
    low = s.lower()
    crop = None
    for c in ["tomato","potato","corn","maize","apple","grape","cassava","coffee",
              "pepper","bell pepper","peach","cherry","strawberry","soybean",
              "squash","orange","blueberry","raspberry","pear","rice","wheat"]:
        if c in low: crop = c; break
    healthy = "healthy" in low
    disease = "healthy" if healthy else s
    return crop, disease

rows=[]
for name, root in PRESENT.items():
    labels = sorted({p.parent.name for p in iter_images(root)})
    for lab in labels:
        crop, dis = parse_label(lab)
        rows.append(dict(source_dataset=name, original_label=lab,
                         crop_canonical=crop or "UNKNOWN",
                         disease_canonical=dis,
                         needs_review=(crop is None)))
cw = pd.DataFrame(rows)
cw.to_csv(DATA_MAPPING/"class_crosswalk.csv", index=False)
print(f"crosswalk rows: {len(cw)} | need manual review: {cw['needs_review'].sum()}")
display(cw.head(20))

In [ ]:
# Seed the disease->pathogen table the harm matrix needs (manual column left blank)
seed = (cw[["source_dataset","crop_canonical","disease_canonical"]]
        .drop_duplicates()
        .assign(eppo_code="", pathogen_type="", action_class="  # FILL MANUALLY"))
seed.to_csv(DATA_MAPPING/"disease_pathogen.csv", index=False)
print("wrote data/mapping/disease_pathogen.csv — fill pathogen_type + action_class by hand,")
print("using EPPO / AGROVOC / UC IPM as authoritative sources (see DATASETS.md §6).")

---
### For the paper's *Threats to validity*
- State the split was **leaf-grouped** and quote the random-split leak fraction.
- Report the **cross-dataset near-duplicate count** and confirm those images were
  removed from evaluation sets before any generalization number was computed.
- Note that the class crosswalk was seeded automatically but **manually verified**;
  report how many labels needed correction.

**Next:** `05_baseline_pretraining.ipynb`.